# 🚀 GPT-CUDA v2: Fixed Transformer from Scratch

**Run this notebook on a CUDA GPU** (Colab T4, Thunder Compute, etc.)

### Fixes applied over v1:
1. 🔴 **ReLU/GELU actually applied** in LinearLayer (was commented out)
2. 🔴 **RMSNorm gamma learnable** with Adam (was frozen at 1.0)
3. 🟠 **Xavier/Glorot init** (was uniform [-0.05, 0.05])
4. 🟡 **LR warmup + cosine decay** (was constant LR)
5. 🟢 **GELU instead of ReLU** in FFN (modern standard)

**Expected: loss should decrease from ~4.0 to ~0.5-1.0 over 10K steps**

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Download training data (Shakespeare)
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt
!wc -c input.txt
print('✓ Data downloaded')

In [ ]:
# Compile the project
!nvcc -O3 -o gpt_cuda main.cu GPTModel.cu TransformerBlock.cu AttentionLayer.cu FeedForwardLayer.cu RMSNormLayer.cu EmbeddingLayer.cu LinearLayer.cu DataLoader.cpp -lcublas 2>&1
print('✓ Compilation complete')

In [ ]:
# Train!
!./gpt_cuda

In [ ]:
# Plot training loss
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('training_log.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(df['iteration'], df['loss'])
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(df['iteration'], df['lr'])
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Learning Rate')
ax2.set_title('LR Schedule (warmup + cosine decay)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Training curve saved as training_curve.png')